In [ ]:
import os
import glob
import datetime
import numpy as np
from scipy import stats
from scipy.stats import ttest_ind
import statsmodels.stats.multitest as smm
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
import plotnine as p9

# Figure 1

Bacillota reference genomes were collected and subsampled as described in the text. The following SBATCH was used to prepare a concatenated core gene tree from these reference genomes. The treefile was imported to iToL where it was subsequently annotated and decorated. 

In [ ]:
#!/bin/bash
#SBATCH -J make_tree
#SBATCH -A account
#SBATCH -N 1 --ntasks-per-node=10
#SBATCH --mem=128G
#SBATCH -t 52:00:00
#SBATCH -p default

module load apptainer

cd ${SLURM_SUBMIT_DIR}

export GTDBTK_DATA_PATH=path/to/release232
s1="path/to/gtdbtk_2.7.2.sif"
s2="path/to/iqtree2_2.4.0.sif"

apptainer exec --bind ${GTDBTK_DATA_PATH}:/refdata ${s1} \
 gtdbtk identify --genome_dir ${indir} --out_dir ${out1} \
 --cpus 10 --extension fna

apptainer exec --bind ${GTDBTK_DATA_PATH}:/refdata ${s1} \
 gtdbtk align --identify_dir ${out1} --out_dir ${out2} \
 --cpus 20 --skip_gtdb_refs

gunzip ${out2}/align/gtdbtk.bac120.user_msa.fasta.gz

apptainer exec ${s2} \
 iqtree2 -s ${out2}/align/gtdbtk.bac120.user_msa.fasta \
 -m LG+F+R6 -B 1000 --bnni -nt AUTO

Genes and pathways of interest were visualized from the annotation approaches described in the text.

In [ ]:
df = pd.read_excel("Figure1_data.xlsx", sheet_name="MCCA_producers_pwys",index_col=0)

fig, ax = plt.subplots(figsize=(8, 6))

AAred_cols = df.columns[7:11]

for i, row in enumerate(df.index):
    for j, col in enumerate(df.columns):

        marker = "s" if col in AAred_cols else "o"

        if df.loc[row, col] == 1:
            ax.scatter(j, i,
                       marker=marker,
                       s=275,
                       facecolors="black",
                       edgecolors="white")
        else:
            ax.scatter(j, i,
                       marker=marker,
                       s=275,
                       facecolors="white",
                       edgecolors="white")

ax.set_xticks(range(len(df.columns)))
ax.set_xticklabels(df.columns, rotation=90)
ax.xaxis.tick_top()
ax.xaxis.set_label_position('top')
ax.set_yticks(range(len(df.index)))
ax.set_yticklabels(df.index)
ax.invert_yaxis()

plt.tight_layout()